<a href="https://colab.research.google.com/github/figaz123/SpeechRecognitionForMeeting/blob/main/Speech_recognition_for_Meeting_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 Pipeline Transkripsi Audio Rapat (Speech-to-Text)
Notebook ini telah disederhanakan untuk memproses rekaman audio rapat menjadi transkrip teks mentah (*raw transcript*) secara akurat menggunakan OpenAI Whisper.

## **Tahap 1: Instalasi & Impor Dependensi**
Bagian ini akan memasang modul OpenAI Whisper dan mengimpor pustaka Python yang dibutuhkan.

In [3]:
#@title ⚙️ Langkah 1: Hubungkan Drive & Lihat Daftar File { display-mode: "form" }

import os
from google.colab import drive

# 1. Mount Google Drive otomatis
if not os.path.exists('/content/drive'):
    print("🔗 Menghubungkan ke Google Drive...")
    drive.mount('/content/drive')

#@markdown ### 📂 Atur Folder Penyimpanan Anda
drive_folder = "/content/drive/MyDrive/voice meeting/voice meeting" #@param {type:"string"}

# 2. Ambil dan tampilkan list file di folder tersebut
print("\n" + "="*50)
print("📂 DAFTAR FILE AUDIO DI FOLDER ANDA")
print("="*50)

if os.path.exists(drive_folder):
    print(f"✅ Folder aktif: {drive_folder}\n")
    try:
        all_files = os.listdir(drive_folder)
        audio_extensions = ('.mp3', '.wav', '.m4a', '.mp4', '.ogg', '.flac')
        audio_files = [f for f in all_files if f.lower().endswith(audio_extensions)]

        if audio_files:
            print("Silakan salin salah satu nama file di bawah untuk dimasukkan pada Langkah 2:")
            for file in sorted(audio_files):
                print(f"   • {file}")
        else:
            print("❌ Tidak ada file audio (.mp3, .wav, .m4a, dll.) di folder ini.")
    except Exception as e:
        print(f"❌ Gagal membaca isi folder: {e}")
else:
    print(f"❌ Folder tidak ditemukan! Pastikan path '{drive_folder}' sudah benar.")


📂 DAFTAR FILE AUDIO DI FOLDER ANDA
✅ Folder aktif: /content/drive/MyDrive/voice meeting/voice meeting

Silakan salin salah satu nama file di bawah untuk dimasukkan pada Langkah 2:
   • FGD 5.m4a
   • Recording (67).m4a
   • Recording (74).m4a
   • Recording (75).m4a
   • Recording (77).m4a
   • Recording (78).m4a
   • Untitled video - Made with Clipchamp.m4a
   • Video Project 1.m4a
   • Video Project 11.m4a
   • Video Project 3.m4a
   • Video Project 5.m4a
   • Voice 017-1.m4a
   • audio1895607198.m4a
   • rapat_SDM.m4a


In [2]:
#@title Instalasi Modul (Whisper & FFmpeg) { display-mode: "form" }
import sys
import os
import shutil
from google.colab import drive, files

print("Memasang OpenAI Whisper dan dependensi... (Silakan tunggu)")
!pip install -q git+https://github.com/openai/whisper.git
!apt-get install -y ffmpeg -q > /dev/null
print("✅ Instalasi selesai dengan sukses!")

Memasang OpenAI Whisper dan dependensi... (Silakan tunggu)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Instalasi selesai dengan sukses!


## **Tahap 2: Konfigurasi & Input File Audio**
Hubungkan Google Drive Anda dan atur lokasi file audio rapat yang ingin ditranskripsikan.

In [4]:
#@title 🎯 Langkah 2: Pilih File & Atur Parameter Whisper { display-mode: "form" }

#@markdown ### 📝 Ketik Nama File yang Ingin Ditranskrip (dari list di atas)
audio_filename = "Video Project 11.m4a" #@param {type:"string"}

#@markdown ### 🛠️ Atur Parameter Model
whisper_model = "large" #@param ["tiny", "base", "small", "medium", "large"] {type:"string"}
language = "id" #@param ["id", "en", "auto"] {type:"string"}

# Memastikan variabel folder dari sel sebelumnya sudah didefinisikan
try:
    full_audio_path = os.path.join(drive_folder, audio_filename)

    print("===================== VALIDASI PILIHAN =====================")
    if os.path.exists(full_audio_path):
        print(f"✅ BERHASIL: File '{audio_filename}' ditemukan dan siap ditranskripsi.")
        print(f"⚙️ Model: {whisper_model} | Bahasa: {language}")
        print(f"👉 Silakan lanjut jalankan 'Tahap 3: Eksekusi Transkripsi' di bawah.")
    else:
        print(f"❌ ERROR: File '{audio_filename}' tidak ditemukan di folder '{drive_folder}'.")
        print("Silakan periksa kembali daftar file di atas dan pastikan ejaan nama file sudah tepat.")
except NameError:
    print("❌ ERROR: Silakan jalankan 'Langkah 1: Hubungkan Drive & Lihat Daftar File' terlebih dahulu!")

===================== VALIDASI PILIHAN =====================
✅ BERHASIL: File 'Video Project 11.m4a' ditemukan dan siap ditranskripsi.
⚙️ Model: large | Bahasa: id
👉 Silakan lanjut jalankan 'Tahap 3: Eksekusi Transkripsi' di bawah.


## **Tahap 3: Eksekusi Transkripsi (Speech-to-Text)**
Jalankan mesin Whisper untuk mengubah rekaman audio menjadi teks.

In [5]:
#@title Mulai Proses Transkripsi { display-mode: "form" }

if not os.path.exists(full_audio_path):
    print("Proses dibatalkan karena file input tidak valid.")
else:
    print("Memulai transkripsi menggunakan Whisper... (Proses ini memakan waktu sesuai durasi audio)")

    # Tentukan parameter bahasa jika bukan auto
    lang_param = f"--language {language}" if language != "auto" else ""

    # Eksekusi command line whisper
    !whisper "{full_audio_path}" --model {whisper_model} {lang_param} --output_format txt --output_dir .

    # Menentukan nama output file default dari whisper
    base_name = os.path.splitext(audio_filename)[0]
    whisper_output_txt = f"{base_name}.txt"

    if os.path.exists(whisper_output_txt):
        print("\n✅ Transkripsi selesai dengan sukses!")
    else:
        print("\n❌ Terjadi kesalahan saat generate transkrip.")

Memulai transkripsi menggunakan Whisper... (Proses ini memakan waktu sesuai durasi audio)
100%|█████████████████████████████████████| 2.88G/2.88G [00:33<00:00, 91.7MiB/s]
[00:00.000 --> 00:06.740]  Nah, dulu sempat juga ketika di sparring itu sempat hasilnya cukup jauh.
[00:07.560 --> 00:16.760]  Nah, ketika semakin lama alat tersebut dipakai kan pasti akan semakin mungkin akan perlu diganti ya.
[00:16.860 --> 00:22.060]  Nah, itu biasanya akan terdeteksi di verifikasi tahunan atau kalibrasi tahunan.
[00:22.060 --> 00:29.940]  Nanti dari vendor akan menyarankan, oh ini sensor probe-nya sudah tidak bagus, sudah dicoba berkali-kali namun masih jauh seperti itu.
[00:30.060 --> 00:36.280]  Biasanya mereka akan menyarankan kepada kami untuk melakukan penggantian sensor probe gitu sih.
[00:37.420 --> 00:43.980]  Kalau dari elektrik internal biasanya hanya untuk, ini untuk sparring ya sekali lagi.
[00:44.120 --> 00:46.380]  Karena sparring ini terkoneksi kepada KLHK.
[00:46.380 --> 00:51.720]

## **Tahap 4: Penyimpanan & Unduh Hasil Transkrip**
Simpan hasil transkrip ke Google Drive di folder yang sama dengan file audio aslinya, serta unduh langsung ke komputer Anda.

In [6]:
#@title Simpan & Unduh Hasil Transkrip { display-mode: "form" }

base_name = os.path.splitext(audio_filename)[0]
local_txt = f"{base_name}.txt"

if os.path.exists(local_txt):
    # Copy ke Google Drive
    destination_drive_path = os.path.join(drive_folder, local_txt)
    shutil.copy(local_txt, destination_drive_path)
    print(f"💾 Salinan sukses disimpan di Google Drive: {destination_drive_path}")

    # Pemicu download otomatis ke perangkat lokal
    print("📥 Mengunduh file transkrip ke perangkat Anda...")
    files.download(local_txt)
else:
    print("❌ File transkrip tidak ditemukan. Silakan jalankan Tahap 3 terlebih dahulu.")

💾 Salinan sukses disimpan di Google Drive: /content/drive/MyDrive/voice meeting/voice meeting/Video Project 11.txt
📥 Mengunduh file transkrip ke perangkat Anda...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Selesai**
Proses transkripsi audio rapat menggunakan Whisper telah diselesaikan tanpa menggunakan model AI tambahan.

---